In [1]:
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans, KShape
from tslearn.utils import to_time_series_dataset
import plotly.express as px
import pandas as pd
import numpy as np
import librosa
import time
import json
import os

/home/data/data_mining_unipi/.venv/lib/python3.10/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
# base path
mp3_base_path = "../fedez_fibra/"
dataset_output_dir = "../timeseries_datasets/"
tracks_list = os.listdir(mp3_base_path)


In [3]:
# librosa parameters
# hop length: number of samples between the start of two consecutive windows in the signal. https://learnius.com/slp/4+Speech+Signal+Representations/1+Time-Domain/3+Short-Time+Processing/hop+length

sampling_rate         =  22050 # extracted from mp3
song_start_offset_sec = 30 # to skip intro
window_duration_sec   = 60
hop_length = 512 

start_offset_sec = 30
windows_len_sec  = 60
num_samples_in_window = int(sampling_rate * windows_len_sec)
skipped_tracks = 0

n_fft = 2048  # frame length for spectral features
frame_length = 2048 # for time-domain features

number_tracks = len(tracks_list)
print(f"Found: {number_tracks} mp3 tracks")

Found: 454 mp3 tracks


In [4]:
# silence threshold is 20db
# https://audiology-web.s3.amazonaws.com/migrated/NoiseChart_Poster-%208.5x11.pdf_5399b289427535.32730330.pdf

def find_silence_points(data, sampling_rate, top_db=20):
    num_samples = len(data)
    duration = num_samples / sampling_rate
    
    # intervals where db > top_db
    # intervals[i] == (start_i, end_i) are the start and end time (in samples) of non-silent interval i
    intervals = librosa.effects.split(data, top_db=top_db)
    
    if len(intervals) == 0:
        # track is silent
        return {
            'silence_start_sec': duration,
            'silence_end_sec': 0,
            'silence_middle_total_sec': 0,
            'silence_middle_count': 0,
            'silence_middle_mean_sec': 0,
            'is_silent': True
        }
    
    # silence at the beginning
    first_sound = intervals[0][0]
    silence_start = first_sound / sampling_rate
    
    # silence at end, last non silence sample
    last_sound = intervals[-1][1]
    silence_end = (num_samples - last_sound) / sampling_rate
    
    # silence in the middle
    middle_silences = []
    
    for i in range(len(intervals) - 1):
        gap_start = intervals[i][1]
        gap_end = intervals[i + 1][0]
        gap_duration = (gap_end - gap_start) / sampling_rate
        middle_silences.append(gap_duration)
    
    # statistics for silences
    middle_count = len(middle_silences)
    middle_total = sum(middle_silences)
    middle_mean = middle_total / middle_count if middle_count > 0 else 0
    
    return {
        'silence_start_sec': round(silence_start, 2),
        'silence_end_sec': round(silence_end, 2),
        'silence_middle_total_sec': round(middle_total, 2),
        'silence_middle_count': middle_count,
        'silence_middle_mean_sec': round(middle_mean, 2),
        'is_silent': False
    }

In [5]:

# check if tracks are in the origianl tracks dataset
tracks = pd.read_csv("../original_datasets/tracks.csv")

# parse fname -> artistID - trackID
tracks_info = []
tracks_timeseries = []

start = time.time()

for i in range(number_tracks):
    
    # :-4 take fname without .mp3
    # split on " - " to get left and right part    
    fname = tracks_list[i]
    splitted_fname = fname[:-4].split(" - ")
    artist_id = splitted_fname[0]
    track_id = splitted_fname[1]
    
    # extract same size window len
    data, sr = librosa.load(mp3_base_path+fname, offset=start_offset_sec, duration=windows_len_sec)
    num_samples = data.shape[0]
    
    # skip data if not enough samples
    if num_samples < num_samples_in_window:
        print(f"** Skipping: {track_id} not enough samples: {num_samples}")
        skipped_tracks += 1
        continue
    
    # basic track info to expand with features
    track_info = {"id": track_id, "id_artist": artist_id, "num_samples": num_samples, "sr": sr}
    track_info_ts = {"id": track_id, "id_artist": artist_id, "num_samples": num_samples, "sr": sr}

    # duration ms
    duration_ms = librosa.get_duration(y=data, sr=sr) * 1000
    bpms, beats = librosa.beat.beat_track(y=data, sr=sr)
    track_info['duration_ms'] = duration_ms
    
    # if it returns afloat use it else extract the first element as it is 1 element array
    if isinstance(bpms, float):
        track_info['bpm'] = bpms
    else:
        track_info['bpm'] = bpms[0]
    
    centroid = librosa.feature.spectral_centroid(y=data, sr=sr, hop_length=hop_length, n_fft=n_fft)[0]
    track_info_ts['centroid'] = centroid
    track_info['centroid_mean'] = np.mean(centroid)
    track_info['centroid_std'] = np.std(centroid)
    
    rolloff = librosa.feature.spectral_rolloff(y=data, sr=sr, hop_length=hop_length, n_fft=n_fft)[0]
    track_info_ts['rolloff'] = rolloff
    track_info['rolloff_mean'] = np.mean(rolloff)
    track_info['rolloff_std'] = np.std(rolloff)
    
    flux = librosa.onset.onset_strength(y=data, sr=sr, hop_length=hop_length, n_fft=n_fft)
    track_info_ts['flux'] = flux
    track_info['flux_mean'] = np.mean(flux)
    track_info['flux_std'] = np.std(flux)
    
    rms = librosa.feature.rms(y=data, hop_length=hop_length, frame_length=frame_length)[0]
    track_info_ts['rms'] = rms
    track_info['rms_mean'] = np.mean(rms)
    track_info['rms_std'] = np.std(rms)
    
    zcr = librosa.feature.zero_crossing_rate(y=data, hop_length=hop_length, frame_length=frame_length)[0]
    track_info_ts['zcr'] = zcr
    track_info['zcr_mean'] = np.mean(zcr)
    track_info['zcr_std'] = np.std(zcr)
    
    flatness = librosa.feature.spectral_flatness(y=data, hop_length=hop_length, n_fft=n_fft)[0]
    track_info['flatness_mean'] = np.mean(flatness)
    track_info['flatness_std'] = np.std(flatness)
    
    spec_bw = librosa.feature.spectral_bandwidth(y=data, sr=sr,  hop_length=hop_length, n_fft=n_fft)[0]
    track_info_ts['spectral_bw'] = spec_bw
    track_info['spectral_complexity_mean'] = np.mean(spec_bw)
    track_info['spectral_complexity_std'] = np.std(spec_bw)
    
    # extract fundamental frequency of signal
    # fmin = lowest pitch to detect 50 hz below male voice
    # fmax = highes pitch to detect 500 hz above female voice
    # hop length = how often to estimate pitch
    f0 = librosa.yin(data, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
    
    # Filter out zeros and very low values (unvoiced segments)
    f0_voiced = f0[f0 > 50]
    track_info['pitch_mean'] = np.mean(f0_voiced) if len(f0_voiced) > 0 else 0
    track_info['pitch_std'] = np.std(f0_voiced) if len(f0_voiced) > 0 else 0
    
    loudness_db = librosa.amplitude_to_db(rms, ref=np.max)
    track_info['loudness_mean'] = np.mean(loudness_db)
    track_info['loudness_std'] = np.std(loudness_db)

    # compute silence points and save
    silence_stats = find_silence_points(data, sr, top_db=20)
    track_info.update(silence_stats)
    
    tracks_info.append(track_info)
    tracks_timeseries.append(track_info_ts)
    print(f"Progress {i}/{number_tracks} Data len: {data.size}")

end = time.time()

print(f"Took: {end-start:.2f} Skipped: {skipped_tracks} tracks. Total tracks loaded: {number_tracks-skipped_tracks}")

Progress 0/454 Data len: 1323000
Progress 1/454 Data len: 1323000
Progress 2/454 Data len: 1323000
Progress 3/454 Data len: 1323000
Progress 4/454 Data len: 1323000
Progress 5/454 Data len: 1323000
Progress 6/454 Data len: 1323000
Progress 7/454 Data len: 1323000
Progress 8/454 Data len: 1323000
Progress 9/454 Data len: 1323000
Progress 10/454 Data len: 1323000
Progress 11/454 Data len: 1323000
Progress 12/454 Data len: 1323000
Progress 13/454 Data len: 1323000
Progress 14/454 Data len: 1323000
Progress 15/454 Data len: 1323000
Progress 16/454 Data len: 1323000
Progress 17/454 Data len: 1323000
Progress 18/454 Data len: 1323000
Progress 19/454 Data len: 1323000
Progress 20/454 Data len: 1323000
Progress 21/454 Data len: 1323000
Progress 22/454 Data len: 1323000
Progress 23/454 Data len: 1323000
Progress 24/454 Data len: 1323000
Progress 25/454 Data len: 1323000
** Skipping: TR403785 not enough samples: 394757
Progress 27/454 Data len: 1323000
Progress 28/454 Data len: 1323000
Progress 

/tmp/ipykernel_1941017/1350200425.py:20: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(mp3_base_path+fname, offset=start_offset_sec, duration=windows_len_sec)
/home/data/data_mining_unipi/.venv/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Progress 96/454 Data len: 1323000
Progress 97/454 Data len: 1323000
Progress 98/454 Data len: 1323000
Progress 99/454 Data len: 1323000
Progress 100/454 Data len: 1323000
Progress 101/454 Data len: 1323000
Progress 102/454 Data len: 1323000
Progress 103/454 Data len: 1323000
Progress 104/454 Data len: 1323000
Progress 105/454 Data len: 1323000
Progress 106/454 Data len: 1323000
Progress 107/454 Data len: 1323000
Progress 108/454 Data len: 1323000
Progress 109/454 Data len: 1323000
Progress 110/454 Data len: 1323000
Progress 111/454 Data len: 1323000
Progress 112/454 Data len: 1323000
Progress 113/454 Data len: 1323000
Progress 114/454 Data len: 1323000
Progress 115/454 Data len: 1323000
Progress 116/454 Data len: 1323000
Progress 117/454 Data len: 1323000
Progress 118/454 Data len: 1323000
Progress 119/454 Data len: 1323000
Progress 120/454 Data len: 1323000
Progress 121/454 Data len: 1323000
Progress 122/454 Data len: 1323000
Progress 123/454 Data len: 1323000
Progress 124/454 Data le

In [6]:
tracks_info_df = pd.DataFrame(tracks_info)
tracks_info_df.to_csv(dataset_output_dir+"tracks.csv", index=False)

tracks_ts_df = pd.DataFrame(tracks_timeseries)

# convert to json for csv dump
for col in ['centroid', 'rolloff', 'flux', 'rms', 'zcr', 'spectral_bw']:
    tracks_ts_df[col] = tracks_ts_df[col].apply(lambda x: json.dumps(x.tolist()))
    
tracks_ts_df.to_csv(dataset_output_dir+"tracks_timeseries.csv", index=False)

In [7]:
# extract artists id and correlate it to name
unique_artists = tracks_info_df.id_artist.unique()
artists_names = list(tracks[tracks['id_artist'].isin(unique_artists)]['name_artist'].unique())

In [8]:
# check how many tracks are in the tracks df
track_matches = tracks_info_df['id'].isin(tracks['id']).sum()
missing_tracks_count = number_tracks - track_matches

print(f"Total mp3 tracks: {number_tracks} Matches found: {track_matches} Missing tracks: {missing_tracks_count}")

print(f"Artists names: {', '.join(artists_names)}")

Total mp3 tracks: 454 Matches found: 440 Missing tracks: 14
Artists names: Fabri Fibra, Fedez
